# BÀI THỰC HÀNH SỐ 3 - MONTE CARLO METHODS TRONG HỌC TĂNG CƯỜNG

**Học phần:** Học tăng cường - Reinforcement Learning  |  **Môi trường chính:** `Blackjack-v1`  |  **Môi trường phụ:** `FrozenLake-v1`

Notebook này gom toàn bộ 36 bài. Các hàm Monte Carlo (tự cài đặt, không dùng thư viện RL) nằm trong `../src/mc_utils.py`;
mỗi bài cũng chạy độc lập bằng `python src/baiXX.py`.

In [ ]:
# Google Colab: bỏ comment dòng dưới để cài thư viện, và upload thư mục Lab03/src lên cùng cấp với notebook.
# !pip install -q "gymnasium[toy-text,classic-control]"
import sys
from pathlib import Path

SRC_DIR = Path("..") / "src"                 # notebook nằm trong Lab03/notebooks/
if not SRC_DIR.exists():
    SRC_DIR = Path("src")                    # hoặc chạy từ thư mục Lab03/
sys.path.insert(0, str(SRC_DIR.resolve()))

import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

print("Gymnasium:", gym.__version__)
print("NumPy:", np.__version__)

STUDENT_ID = "MSSV"
STUDENT_NAME = "Ho Ten"

MAIN_ENV = "Blackjack-v1"

OPTIONAL_ENVS = [
    "FrozenLake-v1",
]

---
# PHẦN A - Làm quen với episodic environment

### Bài 1. Tạo Blackjack-v1

In [ ]:
import gymnasium as gym

env = gym.make("Blackjack-v1")
observation, info = env.reset(seed=42)

print("observation      :", observation)
print("info             :", info)
print("observation_space:", env.observation_space)
print("action_space     :", env.action_space)
env.close()

### Bài 2. Phân tích observation

`observation = (player_sum, dealer_card, usable_ace)`. Gymnasium trả `usable_ace` dạng 0/1; `0 == False` nên tra cứu dict bằng `(20, 10, False)` vẫn đúng.

In [ ]:
import gymnasium as gym

env = gym.make("Blackjack-v1")

# observation = (player_sum, dealer_card, usable_ace)
#   player_sum  : tổng điểm bài hiện tại của người chơi (Ace tính 11 nếu không bị quá 21).
#   dealer_card : lá bài dealer đang ngửa (1 = Ace, 2..10; J/Q/K tính là 10).
#   usable_ace  : người chơi có Ace đang được tính là 11 hay không
#                 (Gymnasium trả về 0/1 nên in ra dạng bool cho dễ đọc).
observations = []
for episode_id in range(12):                       # 12 episode khác nhau
    obs, _ = env.reset(seed=episode_id)
    observations.append(obs)

for i, obs in enumerate(observations, start=1):
    player_sum, dealer_card, usable_ace = obs
    print(f"{i:2d}. obs={obs} -> player_sum={player_sum:2d}, "
          f"dealer_card={dealer_card:2d}, usable_ace={bool(usable_ace)}")
env.close()

### Bài 3. Phân tích action

In [ ]:
import gymnasium as gym

env = gym.make("Blackjack-v1")
print("action_space:", env.action_space, "| số action:", env.action_space.n)

# Action của Blackjack-v1: 0 = stick (dừng, không rút thêm), 1 = hit (rút thêm một lá).
ACTION_NAMES = {
    0: "stick (dừng)",
    1: "hit (rút bài)",
}
for action_id, name in ACTION_NAMES.items():
    print(f"action {action_id} = {name}")
env.close()

### Bài 4. Một episode ngẫu nhiên (random policy)

In [ ]:
import gymnasium as gym

from mc_utils import BLACKJACK_ACTION_NAMES

env = gym.make("Blackjack-v1")
env.action_space.seed(0)

state, _ = env.reset(seed=0)
done, step = False, 0
while not done:
    action = env.action_space.sample()             # random policy
    next_state, reward, terminated, truncated, _ = env.step(action)
    step += 1
    print(f"Bước {step}: state={state}, action={BLACKJACK_ACTION_NAMES[action]}, "
          f"reward={reward}, next_state={next_state}, "
          f"terminated={terminated}, truncated={truncated}")
    state = next_state
    done = terminated or truncated
env.close()

### Bài 5. Lưu episode: generate_episode(env, policy, seed)

In [ ]:
import gymnasium as gym

from mc_utils import generate_episode, make_random_policy   # generate_episode nằm trong mc_utils.py

env = gym.make("Blackjack-v1")
random_policy = make_random_policy(env.action_space.n, seed=0)

# Mỗi phần tử của episode: (state, action, reward)
episode = generate_episode(env, random_policy, seed=1)
print("Số bước:", len(episode))
for t, (state, action, reward) in enumerate(episode):
    print(f"t={t}: state={state}, action={action}, reward={reward}")
env.close()

---
# PHẦN B - Return từ episode

### Bài 6. Chuỗi reward

In [ ]:
import gymnasium as gym

from mc_utils import generate_episode, make_random_policy

env = gym.make("Blackjack-v1")
episode = generate_episode(env, make_random_policy(env.action_space.n, seed=0), seed=1)

rewards = [reward for _, _, reward in episode]
print("Episode length:", len(episode))
print("Rewards       :", rewards)
print("Total reward  :", sum(rewards))
env.close()

### Bài 7. Tính return từ cuối episode

Quy ước: `rewards[t] = R_(t+1)`, nên `G_t = rewards[t] + gamma * G_(t+1)`, duyệt ngược từ cuối episode.

In [ ]:
import gymnasium as gym

from mc_utils import compute_returns, generate_episode, make_random_policy

# compute_returns duyệt ngược: G_t = R_{t+1} + gamma * G_{t+1}  (xem mc_utils.py)
env = gym.make("Blackjack-v1")
episode = generate_episode(env, make_random_policy(env.action_space.n, seed=0), seed=1)
rewards = [reward for _, _, reward in episode]

for gamma in (1.0, 0.9):
    print(f"gamma={gamma}: rewards={rewards} -> returns={compute_returns(rewards, gamma)}")
env.close()

### Bài 8. Kiểm tra return với episode giả

In [ ]:
import numpy as np

from mc_utils import compute_returns

rewards = [0, 0, 1]          # R1=0, R2=0, R3=1
for gamma in (1.0, 0.9, 0.5):
    G = compute_returns(rewards, gamma)
    # Kết quả mong đợi: [gamma^2, gamma, 1]
    expected = [gamma ** 2, gamma, 1.0]
    assert np.allclose(G, expected), (G, expected)
    print(f"gamma={gamma}: G = {G}  (đúng với [gamma^2, gamma, 1])")

### Bài 9. Gắn return vào trajectory: (state, action, reward, G_t)

In [ ]:
import gymnasium as gym

from mc_utils import compute_returns, generate_episode, make_random_policy

env = gym.make("Blackjack-v1")
episode = generate_episode(env, make_random_policy(env.action_space.n, seed=0), seed=3)

returns = compute_returns([r for _, _, r in episode], gamma=1.0)
episode_with_return = [(s, a, r, G) for (s, a, r), G in zip(episode, returns)]

for t, step in enumerate(episode_with_return):
    print(f"t={t}: (state, action, reward, G_t) = {step}")
env.close()

### Bài 10. So sánh gamma: mean G_0 theo gamma

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from mc_utils import compute_returns, generate_episode, make_random_policy, save_figure

env = gym.make("Blackjack-v1")
policy = make_random_policy(env.action_space.n, seed=0)    # cùng một policy cho mọi gamma
n_episodes = 5000

# Sinh episode một lần, rồi tính return với từng gamma để so sánh công bằng.
all_rewards = [[r for _, _, r in generate_episode(env, policy, seed=0 if i == 0 else None)]
               for i in range(n_episodes)]

gammas = [0.5, 0.8, 0.9, 0.99, 1.0]
mean_g0 = []
for gamma in gammas:
    g0 = [compute_returns(rewards, gamma)[0] for rewards in all_rewards]
    mean_g0.append(float(np.mean(g0)))
    print(f"gamma={gamma:<5}: mean G_0 = {mean_g0[-1]:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(gammas, mean_g0, marker="o")
ax.set_xlabel("gamma")
ax.set_ylabel("mean G_0")
ax.set_title("Bài 10: mean G_0 theo gamma (random policy, Blackjack)")
ax.grid(alpha=0.3)
save_figure(fig, "gamma_comparison.png")
env.close()

---
# PHẦN C - Monte Carlo Prediction cơ bản

### Bài 11. Policy cố định: stick on 20

In [ ]:
from mc_utils import BLACKJACK_ACTION_NAMES


def stick_on_20_policy(state):
    """Stick (0) nếu player_sum >= 20, ngược lại hit (1)."""
    player_sum, dealer_card, usable_ace = state

    if player_sum >= 20:
        return 0
    return 1


# Kiểm tra lại ý nghĩa action trước khi dùng: 0 = stick, 1 = hit (xem Bài 3).
for state in [(21, 5, False), (20, 10, False), (19, 10, False), (12, 6, True)]:
    action = stick_on_20_policy(state)
    print(f"state={state} -> action {action} = {BLACKJACK_ACTION_NAMES[action]}")

### Bài 12. Sinh 100 episode với policy stick-on-20 và thống kê win/loss/draw

Reward cuối episode: +1 = win, -1 = loss, 0 = draw.

In [ ]:
import gymnasium as gym

from mc_utils import generate_episode, stick_on_20_policy

env = gym.make("Blackjack-v1")
n_episodes = 100
win = loss = draw = 0

for i in range(n_episodes):
    episode = generate_episode(env, stick_on_20_policy, seed=0 if i == 0 else None)
    total_reward = sum(r for _, _, r in episode)
    if total_reward > 0:
        win += 1
    elif total_reward < 0:
        loss += 1
    else:
        draw += 1

print(f"win : {win:3d} ({win / n_episodes:.1%})")
print(f"loss: {loss:3d} ({loss / n_episodes:.1%})")
print(f"draw: {draw:3d} ({draw / n_episodes:.1%})")
env.close()

### Bài 13. Thu thập return theo state (chưa phân biệt first/every visit)

In [ ]:
from collections import defaultdict

import gymnasium as gym

from mc_utils import compute_returns, generate_episode, stick_on_20_policy

env = gym.make("Blackjack-v1")
returns = defaultdict(list)          # state -> danh sách return quan sát được

for i in range(1000):
    episode = generate_episode(env, stick_on_20_policy, seed=0 if i == 0 else None)
    G = compute_returns([r for _, _, r in episode], gamma=1.0)
    for t, (state, _, _) in enumerate(episode):
        returns[state].append(G[t])  # lưu return tại MỌI lần state xuất hiện

print("Số state đã gặp:", len(returns))
for state in sorted(returns, key=lambda s: -len(returns[s]))[:5]:
    print(f"state={state}: {len(returns[state])} return, 5 đầu = {returns[state][:5]}")
env.close()

### Bài 14. Ước lượng V(s) = trung bình return

In [ ]:
from collections import defaultdict

import gymnasium as gym
import numpy as np

from mc_utils import compute_returns, generate_episode, stick_on_20_policy

env = gym.make("Blackjack-v1")
returns = defaultdict(list)
for i in range(10000):
    episode = generate_episode(env, stick_on_20_policy, seed=0 if i == 0 else None)
    G = compute_returns([r for _, _, r in episode], gamma=1.0)
    for t, (state, _, _) in enumerate(episode):
        returns[state].append(G[t])

V = {}
for state in returns:
    V[state] = np.mean(returns[state])

print("V(s) của 10 state xuất hiện nhiều nhất:")
for state in sorted(V, key=lambda s: -len(returns[s]))[:10]:
    print(f"  V{state} = {V[state]:+.3f}  (n={len(returns[state])})")
env.close()

### Bài 15. Theo dõi estimate V(s) của một state cụ thể khi tăng số episode

Estimate dao động mạnh khi ít episode (state mới gặp vài lần) và dần ổn định khi n tăng.

In [ ]:
from collections import defaultdict

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from mc_utils import compute_returns, generate_episode, save_figure, stick_on_20_policy

env = gym.make("Blackjack-v1")
target_state = (20, 10, False)
checkpoints = [100, 500, 1000, 5000, 10000]

returns = defaultdict(list)
estimates = []
for ep in range(1, max(checkpoints) + 1):
    episode = generate_episode(env, stick_on_20_policy, seed=0 if ep == 1 else None)
    G = compute_returns([r for _, _, r in episode], gamma=1.0)
    for t, (state, _, _) in enumerate(episode):
        returns[state].append(G[t])
    if ep in checkpoints:
        est = np.mean(returns[target_state]) if returns[target_state] else np.nan
        estimates.append(est)
        print(f"Sau {ep:5d} episode: V{target_state} ≈ {est:+.4f} "
              f"(đã gặp {len(returns[target_state])} lần)")

fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(checkpoints, estimates, marker="o")
ax.set_xscale("log")
ax.set_xlabel("Số episode")
ax.set_ylabel(f"Estimate V{target_state}")
ax.set_title("Bài 15: Convergence curve của một state")
ax.grid(alpha=0.3)
save_figure(fig, "v_target_state_convergence.png")
env.close()

---
# PHẦN D - First-Visit Monte Carlo

### Bài 16. Phát hiện first visit của mỗi state trong một episode

In [ ]:
import gymnasium as gym

from mc_utils import first_visit_indices, generate_episode, make_random_policy, stick_on_20_policy

# first_visit_indices(states) (trong mc_utils.py) duyệt episode theo thứ tự thời gian
# và ghi lại chỉ số t đầu tiên của mỗi state - không dùng thư viện RL nào.

# 1) Blackjack: player_sum chỉ tăng nên state gần như không lặp lại trong một episode.
bj = gym.make("Blackjack-v1")
episode = generate_episode(bj, stick_on_20_policy, seed=5)
states = [s for s, _, _ in episode]
print("Blackjack states:", states)
print("First-visit idx :", first_visit_indices(states))
bj.close()

# 2) FrozenLake (random policy): state lặp lại nhiều lần -> first/every visit khác nhau.
fl = gym.make("FrozenLake-v1")
policy = make_random_policy(fl.action_space.n, seed=0)
for seed in range(50):                                   # tìm một episode có state lặp lại
    episode = generate_episode(fl, policy, seed=seed)
    states = [s for s, _, _ in episode]
    if len(set(states)) < len(states):
        break
first = first_visit_indices(states)
print("\nFrozenLake states      :", states)
print("First-visit idx        :", first)
for s in first:
    all_idx = [t for t, x in enumerate(states) if x == s]
    print(f"  state {s}: first-visit t={first[s]}, every-visit t={all_idx}")
fl.close()

### Bài 17. First-Visit MC Prediction

In [ ]:
import gymnasium as gym

from mc_utils import first_visit_mc_prediction, stick_on_20_policy

env = gym.make("Blackjack-v1")
V, returns_count = first_visit_mc_prediction(env, stick_on_20_policy, n_episodes=10000,
                                             gamma=1.0, seed=0)

print("Số state ước lượng được:", len(V))
for state in sorted(V, key=lambda s: -returns_count[s])[:10]:
    print(f"  V{state} = {V[state]:+.3f}  (n={returns_count[state]})")
env.close()

### Bài 18. Chạy First-Visit MC với 100 / 1000 / 10000 / 50000 episode

In [ ]:
import gymnasium as gym

from mc_utils import first_visit_mc_prediction, stick_on_20_policy

env = gym.make("Blackjack-v1")
for n_episodes in [100, 1000, 10000, 50000]:
    V, counts = first_visit_mc_prediction(env, stick_on_20_policy, n_episodes, seed=0)
    print(f"{n_episodes:6d} episode -> {len(V):3d} state được ước lượng, "
          f"tổng số lần cập nhật = {sum(counts.values())}")
env.close()

### Bài 19. Visualize V(s) của Blackjack (heatmap player_sum x dealer_card)

In [ ]:
import gymnasium as gym

from mc_utils import (blackjack_grid, first_visit_mc_prediction, plot_blackjack_grid,
                      stick_on_20_policy)

env = gym.make("Blackjack-v1")
V, _ = first_visit_mc_prediction(env, stick_on_20_policy, n_episodes=200_000, seed=0)

for usable_ace, name in [(False, "no_usable_ace"), (True, "usable_ace")]:
    grid = blackjack_grid(V, usable_ace)
    if not usable_ace:                       # in bảng text đơn giản cho usable_ace = False
        print("V(s), usable_ace=False  (hàng: player_sum 12..21, cột: dealer A,2..10)")
        for ps, row in zip(range(12, 22), grid):
            print(f"{ps:2d} | " + " ".join(f"{v:+.2f}" for v in row))
    plot_blackjack_grid(grid, f"V(s) - stick on 20 - usable_ace={usable_ace}",
                        f"blackjack_value_{name}.png", vmin=-1, vmax=1)
env.close()

---
# PHẦN E - Every-Visit Monte Carlo

### Bài 20. Every-Visit MC Prediction

In [ ]:
import gymnasium as gym

from mc_utils import every_visit_mc_prediction, stick_on_20_policy

env = gym.make("Blackjack-v1")
V, returns_count = every_visit_mc_prediction(env, stick_on_20_policy, n_episodes=10000,
                                             gamma=1.0, seed=0)

print("Số state ước lượng được:", len(V))
for state in sorted(V, key=lambda s: -returns_count[s])[:10]:
    print(f"  V{state} = {V[state]:+.3f}  (n={returns_count[state]})")
env.close()

### Bài 21. So sánh First-Visit và Every-Visit (cùng policy, gamma, n_episodes, seed)

In [ ]:
import gymnasium as gym

from mc_utils import every_visit_mc_prediction, first_visit_mc_prediction, stick_on_20_policy

env = gym.make("Blackjack-v1")
kwargs = dict(policy=stick_on_20_policy, n_episodes=50000, gamma=1.0, seed=0)
V_first, count_first = first_visit_mc_prediction(env, **kwargs)
V_every, count_every = every_visit_mc_prediction(env, **kwargs)

common = [s for s in V_first if s in V_every]
top10 = sorted(common, key=lambda s: -count_first[s])[:10]
print(f"{'state':<18}{'V_first':>10}{'V_every':>10}{'n_first':>9}{'n_every':>9}")
for s in top10:
    print(f"{str(s):<18}{V_first[s]:>+10.4f}{V_every[s]:>+10.4f}"
          f"{count_first[s]:>9}{count_every[s]:>9}")
# Nhận xét: trong Blackjack một state hầu như không lặp lại trong cùng episode
# nên first-visit và every-visit cho kết quả giống nhau (xem Bài 22).
env.close()

### Bài 22. Sai khác giữa First-Visit và Every-Visit

Trong Blackjack, state không lặp lại trong cùng một episode (player_sum chỉ tăng) nên First-Visit và Every-Visit **trùng nhau**. Để thấy sự khác biệt, dùng thêm `FrozenLake-v1` (state lặp lại nhiều lần).

In [ ]:
import gymnasium as gym
import numpy as np

from mc_utils import (every_visit_mc_prediction, first_visit_mc_prediction,
                      make_random_policy, stick_on_20_policy)


def mean_abs_difference(V_first, V_every):
    """Mean |V_first[s] - V_every[s]| trên các state chung."""
    common = [s for s in V_first if s in V_every]
    diffs = [abs(V_first[s] - V_every[s]) for s in common]
    return float(np.mean(diffs)), common


# --- Blackjack ---
env = gym.make("Blackjack-v1")
kw = dict(n_episodes=50000, gamma=1.0, seed=0)
V_f, _ = first_visit_mc_prediction(env, stick_on_20_policy, **kw)
V_e, _ = every_visit_mc_prediction(env, stick_on_20_policy, **kw)
mad, common = mean_abs_difference(V_f, V_e)
print(f"Blackjack : {len(common)} state chung, mean |V_first - V_every| = {mad:.6f}")
env.close()

# --- FrozenLake (state có lặp lại trong episode nên hai phương pháp khác nhau) ---
fl = gym.make("FrozenLake-v1")
kw = dict(n_episodes=20000, gamma=0.95, seed=0)
V_f, _ = first_visit_mc_prediction(fl, make_random_policy(4, seed=1), **kw)
V_e, _ = every_visit_mc_prediction(fl, make_random_policy(4, seed=1), **kw)
mad, common = mean_abs_difference(V_f, V_e)
print(f"FrozenLake: {len(common)} state chung, mean |V_first - V_every| = {mad:.6f}")
for s in sorted(common)[:6]:
    print(f"  state {s:2d}: first={V_f[s]:.4f}, every={V_e[s]:.4f}, |diff|={abs(V_f[s]-V_e[s]):.4f}")
fl.close()

### Bài 23. Biểu đồ hội tụ First-Visit vs Every-Visit

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt

from mc_utils import (make_random_policy, mc_prediction_with_history, save_figure,
                      stick_on_20_policy)

checkpoints = [100, 200, 500, 1000, 2000, 5000, 10000, 20000, 50000]


def run_both(env, policy_factory, states, gamma, n_episodes=50000):
    """Chạy first/every-visit với cùng seed và policy; trả về history của các state."""
    out = {}
    for name, first in [("First-Visit", True), ("Every-Visit", False)]:
        _, _, hist = mc_prediction_with_history(
            env, policy_factory(), n_episodes, gamma, first_visit=first, seed=0,
            track_states=states, checkpoints=checkpoints)
        out[name] = hist
    return out


# --- Hàng 1: Blackjack, 3 state ---
bj = gym.make("Blackjack-v1")
bj_states = [(20, 10, False), (18, 6, False), (13, 2, False)]
bj_hist = run_both(bj, lambda: stick_on_20_policy, bj_states, gamma=1.0)

# --- Hàng 2: FrozenLake, 3 state hay gặp nhất (chọn bằng một lần chạy thử) ---
fl = gym.make("FrozenLake-v1")
fl_policy = lambda: make_random_policy(fl.action_space.n, seed=1)
_, counts, _ = mc_prediction_with_history(fl, fl_policy(), 2000, 0.95, seed=99)
fl_states = sorted(counts, key=lambda s: -counts[s])[:3]
fl_hist = run_both(fl, fl_policy, fl_states, gamma=0.95)

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for row, (title, states, hist) in enumerate([("Blackjack", bj_states, bj_hist),
                                             ("FrozenLake", fl_states, fl_hist)]):
    for col, s in enumerate(states):
        ax = axes[row, col]
        ax.plot(checkpoints, hist["First-Visit"][s], marker="o", label="First-Visit")
        ax.plot(checkpoints, hist["Every-Visit"][s], marker="x", linestyle="--",
                label="Every-Visit")
        ax.set_xscale("log")
        ax.set_title(f"{title}: V{s}")
        ax.set_xlabel("Số episode")
        ax.set_ylabel("Estimate V(s)")
        ax.grid(alpha=0.3)
        ax.legend()
fig.suptitle("First-Visit vs Every-Visit MC (Blackjack: trùng nhau; FrozenLake: khác nhau)")
fig.tight_layout()
save_figure(fig, "first_vs_every_visit.png")
bj.close()
fl.close()

---
# PHẦN F - Action-value function Q(s,a)

### Bài 24. Lưu (state, action) -> return

In [ ]:
from collections import defaultdict

import gymnasium as gym

from mc_utils import compute_returns, generate_episode, make_random_policy

env = gym.make("Blackjack-v1")
policy = make_random_policy(env.action_space.n, seed=0)    # random để mọi action đều được thử

returns = defaultdict(list)          # (state, action) -> danh sách return
for i in range(20000):
    episode = generate_episode(env, policy, seed=0 if i == 0 else None)
    G = compute_returns([r for _, _, r in episode], gamma=1.0)
    for t, (state, action, _) in enumerate(episode):
        returns[(state, action)].append(G[t])

print("Số cặp (state, action) đã gặp:", len(returns))
for state in [(20, 10, False), (13, 2, False)]:
    for action in (0, 1):
        r = returns[(state, action)]
        print(f"{state}, action={action}: n={len(r)}")
env.close()

### Bài 25. Ước lượng Q(s,a) bằng MC

In [ ]:
import gymnasium as gym

from mc_utils import BLACKJACK_ACTION_NAMES, make_random_policy, mc_action_value_prediction

env = gym.make("Blackjack-v1")
# Dùng random policy để cả hai action đều được thử ở mọi state (Q[state][action]).
Q = mc_action_value_prediction(env, make_random_policy(env.action_space.n, seed=0),
                               n_episodes=200_000, gamma=1.0, seed=0)

print("Số state có Q:", len(Q))
for state in [(20, 10, False), (18, 6, False), (13, 2, False), (18, 6, True)]:
    q = Q[state]
    print(f"Q{state}: stick={q[0]:+.3f}, hit={q[1]:+.3f}")
env.close()

### Bài 26. Greedy action từ Q

In [ ]:
import gymnasium as gym
import numpy as np

from mc_utils import BLACKJACK_ACTION_NAMES, make_random_policy, mc_action_value_prediction


def greedy_action(Q, state, n_actions):
    """Trả về action có Q(s,a) lớn nhất bằng np.argmax; state chưa gặp -> 0."""
    if state not in Q:
        return 0
    return int(np.argmax(Q[state][:n_actions]))


env = gym.make("Blackjack-v1")
n_actions = env.action_space.n
Q = mc_action_value_prediction(env, make_random_policy(n_actions, seed=0),
                               n_episodes=200_000, gamma=1.0, seed=0)
for state in [(20, 10, False), (18, 6, False), (13, 2, False), (18, 6, True)]:
    a = greedy_action(Q, state, n_actions)
    print(f"state={state}: Q={np.round(Q[state], 3)} -> greedy = {BLACKJACK_ACTION_NAMES[a]}")
env.close()

---
# PHẦN G - Epsilon-greedy và Exploration

### Bài 27. Epsilon-greedy policy

In [ ]:
import numpy as np

from mc_utils import epsilon_greedy_action

# epsilon_greedy_action (mc_utils.py):
#   - xác suất epsilon    : chọn ngẫu nhiên đều trong mọi action (khám phá)
#   - xác suất 1 - epsilon: chọn greedy action theo Q (khai thác)
rng = np.random.default_rng(0)
Q = {"s": np.array([0.2, 0.8])}          # state giả "s", action 1 là tốt nhất
for epsilon in (0.0, 0.5, 1.0):
    actions = [epsilon_greedy_action(Q, "s", 2, epsilon, rng) for _ in range(10)]
    print(f"epsilon={epsilon}: {actions}")

### Bài 28. Kiểm tra epsilon-greedy với Q giả = [1.0, 5.0], epsilon = 0.1

In [ ]:
from collections import Counter

import numpy as np

from mc_utils import epsilon_greedy_action

rng = np.random.default_rng(0)
Q = {"s": np.array([1.0, 5.0])}
epsilon, n = 0.1, 10000

counts = Counter(epsilon_greedy_action(Q, "s", 2, epsilon, rng) for _ in range(n))
print(f"action 0: {counts[0]} ({counts[0] / n:.3f})")
print(f"action 1 (greedy): {counts[1]} ({counts[1] / n:.3f})")
# Nhận xét: kỳ vọng P(greedy) = 1 - epsilon + epsilon/2 = 0.95, P(action 0) = epsilon/2 = 0.05.
print("Lý thuyết: P(greedy) =", 1 - epsilon + epsilon / 2)

### Bài 29. So sánh epsilon: tần suất chọn greedy action + learning curve

Hình bên phải là reward *lúc train* (còn exploration), nên thấp hơn reward lúc đánh giá greedy.

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from mc_utils import epsilon_greedy_action, moving_average, on_policy_mc_control, save_figure

epsilons = [0.01, 0.05, 0.10, 0.20, 0.50]
rng = np.random.default_rng(0)
Q = {"s": np.array([1.0, 5.0])}

# (a) Tần suất chọn greedy action với Q giả
freq = []
for eps in epsilons:
    actions = [epsilon_greedy_action(Q, "s", 2, eps, rng) for _ in range(10000)]
    freq.append(np.mean(np.array(actions) == 1))
    print(f"epsilon={eps:<5}: tần suất greedy = {freq[-1]:.3f}  (lý thuyết {1 - eps + eps / 2:.3f})")

# (b) Ảnh hưởng của epsilon lên quá trình học (MC control, 100k episode mỗi epsilon)
env = gym.make("Blackjack-v1")
curves = {}
for eps in epsilons:
    _, _, rewards = on_policy_mc_control(env, 100_000, epsilon=eps, seed=42)
    curves[eps] = moving_average(rewards, window=1000)
env.close()

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar([str(e) for e in epsilons], freq)
axes[0].plot([str(e) for e in epsilons], [1 - e + e / 2 for e in epsilons],
             "r.--", label="lý thuyết")
axes[0].set_xlabel("epsilon")
axes[0].set_ylabel("Tần suất chọn greedy action")
axes[0].set_ylim(0, 1.05)
axes[0].legend()
for eps, curve in curves.items():
    axes[1].plot(curve, label=f"epsilon={eps}")
axes[1].set_xlabel("Episode")
axes[1].set_ylabel("Mean reward (moving avg, window=1000)")
axes[1].set_title("Learning curve theo epsilon (reward lúc train, có exploration)")
axes[1].legend()
axes[1].grid(alpha=0.3)
fig.tight_layout()
save_figure(fig, "epsilon_comparison.png")

---
# PHẦN H - On-policy Monte Carlo Control

### Bài 30. Khởi tạo Q, returns_sum, returns_count

In [ ]:
from collections import defaultdict

import gymnasium as gym
import numpy as np

env = gym.make("Blackjack-v1")
n_actions = env.action_space.n

Q = defaultdict(lambda: np.zeros(env.action_space.n))
returns_sum = defaultdict(lambda: np.zeros(n_actions))      # tổng return theo (s, a)
returns_count = defaultdict(lambda: np.zeros(n_actions))    # số lần gặp (s, a)

print("Q ban đầu của một state bất kỳ:", Q[(15, 10, False)])
print("returns_sum:", returns_sum[(15, 10, False)], "| returns_count:", returns_count[(15, 10, False)])
env.close()

### Bài 31. Incremental mean

In [ ]:
from collections import defaultdict

import numpy as np

from mc_utils import update_action_value_incremental

# update_action_value_incremental (mc_utils.py):
#   N(s,a) = N(s,a) + 1;  Q(s,a) = Q(s,a) + (G - Q(s,a)) / N(s,a)
Q = defaultdict(lambda: np.zeros(2))
N = defaultdict(lambda: np.zeros(2))
state, action = (15, 10, False), 1

rng = np.random.default_rng(0)
returns = rng.choice([-1.0, 0.0, 1.0], size=1000)
for G in returns:
    update_action_value_incremental(Q, N, state, action, G)

print("Incremental mean:", Q[state][action])
print("np.mean         :", returns.mean())
assert np.isclose(Q[state][action], returns.mean())
print("-> Hai kết quả bằng nhau, nhưng incremental mean không cần lưu toàn bộ return.")

### Bài 32. On-policy First-Visit MC Control

In [ ]:
import gymnasium as gym

from mc_utils import on_policy_mc_control

# Quy trình (xem on_policy_mc_control trong mc_utils.py):
# khởi tạo Q -> sinh episode bằng epsilon-greedy -> tính return ->
# first-visit update Q(s,a) -> cập nhật policy -> lặp lại.
env = gym.make("Blackjack-v1")
Q, policy, episode_rewards = on_policy_mc_control(env, n_episodes=50000, gamma=1.0,
                                                  epsilon=0.1, seed=42)
print("Số state có Q   :", len(Q))
print("Số state có policy:", len(policy))
print("Số episode_rewards:", len(episode_rewards))
print("Mean reward 1000 episode cuối (lúc train):", sum(episode_rewards[-1000:]) / 1000)
env.close()

### Bài 33. Học policy cho Blackjack (>= 100000 episode)

Policy lấy từ `argmax Q(s,a)`, không hard-code. Kết quả train được cache trong `data/` để Bài 34, 35 dùng lại.

In [ ]:
import numpy as np

from mc_utils import (BLACKJACK_ACTION_NAMES, N_TRAIN, blackjack_grid, plot_blackjack_grid,
                      train_or_load_control)

# Train (kết quả được cache trong data/ để các bài 34, 35 dùng lại).
Q, policy, episode_rewards = train_or_load_control(n_episodes=N_TRAIN, epsilon=0.1)
print(f"Đã train {N_TRAIN} episode, policy có {len(policy)} state.")

# Policy được học từ Q (không hard-code): action = argmax_a Q(s, a).
for state in [(20, 10, False), (18, 6, False), (13, 2, False), (18, 6, True)]:
    action = int(np.argmax(Q[state]))
    print(f"state={state}: Q={np.round(Q[state], 3)} -> policy = {BLACKJACK_ACTION_NAMES[action]}")

# Hình minh họa policy (0 = stick, 1 = hit)
for usable_ace, name in [(False, "no_usable_ace"), (True, "usable_ace")]:
    grid = blackjack_grid(policy, usable_ace)
    plot_blackjack_grid(grid, f"Learned policy (0=stick, 1=hit), usable_ace={usable_ace}",
                        f"blackjack_policy_{name}.png", cmap="coolwarm", vmin=0, vmax=1,
                        colorbar_label="action")

### Bài 34. Đánh giá policy: random vs fixed vs MC learned

Đánh giá trên 100000 episode MỚI (seed khác lúc train), policy greedy hoàn toàn (không explore).

In [ ]:
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from mc_utils import (N_TRAIN, evaluate_policy, make_policy_from_dict, make_random_policy,
                      save_figure, stick_on_20_policy, train_or_load_control)

_, learned_dict, _ = train_or_load_control(n_episodes=N_TRAIN, epsilon=0.1)
learned_policy = make_policy_from_dict(learned_dict)       # chỉ dùng greedy, KHÔNG explore

env = gym.make("Blackjack-v1")
policies = {
    "random": make_random_policy(env.action_space.n, seed=7),
    "fixed (stick on 20)": stick_on_20_policy,
    "MC learned": learned_policy,
}
n_eval = 100_000      # episode MỚI, seed khác lúc train
results = {name: evaluate_policy(env, pol, n_episodes=n_eval, seed=123)
           for name, pol in policies.items()}
env.close()

print(f"{'policy':<22}{'win':>8}{'loss':>8}{'draw':>8}{'mean reward':>14}")
for name, r in results.items():
    print(f"{name:<22}{r['win_rate']:>8.3f}{r['loss_rate']:>8.3f}{r['draw_rate']:>8.3f}"
          f"{r['mean_reward']:>+10.4f} ± {r['std_error']:.4f}")

names = list(results)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(names, [results[n]["mean_reward"] for n in names],
            yerr=[results[n]["std_error"] for n in names], capsize=4)
axes[0].set_ylabel("Mean reward / episode")
axes[0].set_title("Mean reward")
axes[0].axhline(0, color="k", linewidth=0.8)
x = np.arange(len(names))
for i, key in enumerate(["win_rate", "draw_rate", "loss_rate"]):
    axes[1].bar(x + (i - 1) * 0.25, [results[n][key] for n in names], 0.25,
                label=key.replace("_rate", ""))
axes[1].set_xticks(x)
axes[1].set_xticklabels(names)
axes[1].set_title("Win / Draw / Loss rate")
axes[1].legend()
fig.tight_layout()
save_figure(fig, "policy_performance.png")

### Bài 35. Learning curve (moving average, window = 1000)

In [ ]:
import matplotlib.pyplot as plt

from mc_utils import N_TRAIN, moving_average, save_figure, train_or_load_control

_, _, episode_rewards = train_or_load_control(n_episodes=N_TRAIN, epsilon=0.1)
window = 1000
curve = moving_average(episode_rewards, window)

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(range(window, len(episode_rewards) + 1), curve)
ax.set_xlabel("Episode")
ax.set_ylabel(f"Mean reward (moving average, window={window})")
ax.set_title("On-policy MC control - learning curve (Blackjack-v1, epsilon=0.1)")
ax.grid(alpha=0.3)
save_figure(fig, "mc_convergence.png")
print(f"Mean reward {window} episode đầu : {sum(episode_rewards[:window]) / window:+.4f}")
print(f"Mean reward {window} episode cuối: {sum(episode_rewards[-window:]) / window:+.4f}")

---
# PHẦN I - Mini-project

### Bài 36. Monte Carlo Agent hoàn chỉnh (mini-project)

In [ ]:
from main import main

main()

---
# Câu hỏi bắt buộc

**1. Monte Carlo method trong RL là gì?**

Phương pháp ước lượng value function và tìm policy bằng cách **lấy trung bình return** của các episode đã quan sát (được sinh bằng sampling), thay vì dùng công thức Bellman với model đã biết.

**2. Vì sao Monte Carlo được gọi là model-free?**

Vì nó không cần biết transition probability `P(s'|s,a)` hay reward model; chỉ cần chuỗi state, action, reward thu được khi agent tương tác với môi trường.

**3. Vì sao Monte Carlo cần episodic task?**

Vì return `G_t` chỉ tính được đầy đủ khi episode kết thúc. Task không kết thúc (hoặc rất dài) thì không có return hoàn chỉnh, phải đặt `max_steps`/truncate.

**4. Episode khác trajectory như thế nào trong bài này?**

Episode là một lần chạy trọn từ trạng thái đầu đến trạng thái kết thúc. Trajectory là chuỗi `(S_t, A_t, R_{t+1})` ghi lại trong episode đó; trong bài này `generate_episode` trả về chính trajectory của một episode, nên hai khái niệm gần như trùng nhau (episode = đơn vị tương tác, trajectory = dữ liệu ghi lại).

**5. Return được tính khi nào?**

Sau khi episode kết thúc, duyệt ngược từ cuối: `G_t = R_{t+1} + gamma * G_{t+1}`.

**6. Monte Carlo Prediction dùng để làm gì?**

Ước lượng `V_pi(s)` (hoặc `Q_pi(s,a)`) của một policy **cho trước** - tức đánh giá policy đó tốt đến đâu.

**7. First-Visit MC khác Every-Visit MC thế nào?**

First-Visit chỉ dùng return tại lần đầu state xuất hiện trong episode; Every-Visit dùng return ở mọi lần xuất hiện. Cả hai hội tụ về `V_pi`; chỉ khác nhau khi state lặp lại trong episode (Blackjack thì không lặp, FrozenLake thì có).

**8. Vì sao phải lấy trung bình nhiều return?**

Một return là một mẫu ngẫu nhiên (môi trường và policy đều có ngẫu nhiên). Value là **kỳ vọng** của return, nên phải lấy trung bình để xấp xỉ kỳ vọng (luật số lớn).

**9. Khi số episode tăng, estimate V(s) thay đổi như thế nào?**

Dao động nhỏ dần và tiến về giá trị thật `V_pi(s)`; sai số (độ lệch chuẩn của trung bình) giảm theo cỡ `1/sqrt(n)` (xem hình ở Bài 15, 23).

**10. V(s) khác Q(s,a) thế nào?**

`V(s)` là return kỳ vọng khi ở state s rồi theo policy; `Q(s,a)` là return kỳ vọng khi ở s, **chọn action a trước**, rồi mới theo policy. `V(s) = sum_a pi(a|s) Q(s,a)`.

**11. Vì sao MC control cần Q(s,a) thay vì chỉ V(s)?**

Chọn action tốt nhất từ `V(s)` cần biết model (`P` và `R`) để nhìn trước state kế tiếp. Model-free nên phải học trực tiếp giá trị của từng action: `argmax_a Q(s,a)`.

**12. Exploration là gì?**

Thử các action khác ngoài action hiện đang được cho là tốt nhất, để thu thập thông tin về chúng.

**13. Exploitation là gì?**

Chọn action hiện được cho là tốt nhất (greedy) để tối đa hóa reward dựa trên hiểu biết hiện tại.

**14. Epsilon-greedy cân bằng exploration/exploitation thế nào?**

Với xác suất `epsilon` chọn action ngẫu nhiên (explore), với xác suất `1 - epsilon` chọn greedy action (exploit). `epsilon` là núm điều chỉnh sự cân bằng.

**15. Khi epsilon quá lớn thì có vấn đề gì?**

Agent hành động gần như ngẫu nhiên: reward lúc train thấp, học chậm, và policy đang chạy (có exploration) kém xa policy greedy (xem Bài 29).

**16. Khi epsilon quá nhỏ từ đầu thì có vấn đề gì?**

Agent ít thử action khác, Q của các action chưa thử không được cập nhật, dễ kẹt ở policy kém (hội tụ cục bộ) và ước lượng thiếu dữ liệu.

**17. Incremental mean có lợi gì so với lưu toàn bộ returns?**

Cập nhật `Q <- Q + (G - Q)/N` chỉ cần giữ `Q` và `N` (bộ nhớ O(1) mỗi cặp), cho kết quả giống hệt trung bình thường (Bài 31).

**18. On-policy MC control nghĩa là gì?**

Policy dùng để **sinh episode** cũng chính là policy đang được đánh giá và cải thiện (epsilon-greedy theo Q hiện tại).

**19. Policy dùng để sinh episode có thay đổi trong quá trình học không?**

Có. Mỗi lần Q thay đổi, epsilon-greedy theo Q cũng thay đổi, nên episode sau được sinh bởi policy đã cải thiện.

**20. Monte Carlo có bootstrap không?**

Không. MC dùng return thực tế đến hết episode làm target, không dùng ước lượng của state kế tiếp (khác TD và DP).

**21. MC phải đợi đến cuối episode vì sao?**

Vì cần return `G_t` đầy đủ làm target; reward sau thời điểm t chưa biết cho đến khi episode kết thúc.

**22. Random policy và learned policy khác nhau ra sao về mean reward?**

Learned policy có mean reward cao hơn rõ rệt so với random (và fixed) - số liệu cụ thể xem bảng ở Bài 34. Với Blackjack mean reward vẫn thường âm vì luật có lợi cho nhà cái.

**23. Learned policy có chắc chắn tối ưu không? Vì sao?**

Không chắc chắn. Chỉ hội tụ về tối ưu trong giới hạn khi mọi cặp (s,a) được thăm vô hạn lần; thực tế hữu hạn episode, còn nhiễu, và epsilon cố định nghĩa là policy epsilon-greedy chỉ là *epsilon-soft optimal*.

**24. Số episode ảnh hưởng chất lượng estimate thế nào?**

Càng nhiều episode, mỗi state có nhiều mẫu hơn nên phương sai của estimate giảm và nhiều state hiếm được phủ tới (Bài 18: số state ước lượng tăng theo số episode). Ít episode thì estimate nhiễu/thiếu.

**25. Monte Carlo khác Dynamic Programming ở điểm quan trọng nào?**

DP cần model đầy đủ và dùng bootstrap (cập nhật từ ước lượng của state kế tiếp). MC không cần model, học từ episode thực và không bootstrap, nhưng phải đợi hết episode và chỉ dùng cho episodic task.

---
# Nhận xét và kết luận

- Hình lưu ở `Lab03/figures/`: `mc_convergence.png`, `first_vs_every_visit.png`, `policy_performance.png`, `epsilon_comparison.png` (+ heatmap value/policy).
- Điền nhận xét dựa trên **số liệu bạn chạy ra** ở Bài 33-36 (mean reward random / fixed / learned, learning curve).
- Kết luận: Monte Carlo học V(s), Q(s,a) và policy chỉ từ episode, không cần model; cần nhiều episode vì return có phương sai lớn.